# TTS Generator

This notebook generates audio files for `pt1-listening-m1`, `pt1-speaking-m1` using [Kokoro TTS](https://github.com/hexgrad/kokoro) (24 audio files total).

**Voices:** `af_heart` (female), `am_fenrir` (male)

**Instructions:** Select a **GPU runtime** (Runtime > Change runtime type > T4 GPU), then click **Runtime > Run all**. Download the generated .zip files from the last cell.

Generated by `generate_tts_notebook.py`.

In [ ]:
# Install dependencies
!pip install -q kokoro>=0.9.4 soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
import json
import numpy as np
import soundfile as sf
import subprocess
import os
import shutil
import zipfile
from kokoro import KPipeline

pipeline = KPipeline(lang_code='a')
SAMPLE_RATE = 24000
VOICE_MAP = {'female': 'af_heart', 'male': 'am_fenrir'}

ALL_GROUPS = json.loads('''[
  {
    "name": "pt1-listening-m1",
    "blocks": [
      {
        "id": "1-01",
        "output": "1-01.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "1-01.ogg",
            "text": "Do you know if the campus bookstore is still open?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-02",
        "output": "1-02.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-02.ogg",
            "text": "How was your presentation in marketing class?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-03",
        "output": "1-03.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "1-03.ogg",
            "text": "Could you help me move this table to the other room?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-04",
        "output": "1-04.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-04.ogg",
            "text": "I can't seem to log into the university portal."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-05",
        "output": "1-05.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "1-05.ogg",
            "text": "Are you planning to take organic chemistry next semester?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-06",
        "output": "1-06.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-06.ogg",
            "text": "The printer in the study room isn't working again."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-07",
        "output": "1-07.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "1-07.ogg",
            "text": "Did you remember to submit your housing application?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-08",
        "output": "1-08.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-08.ogg",
            "text": "Is the psychology lecture hall easy to find?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-09",
        "output": "1-09.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "1-09-a.ogg",
            "text": "Hey Mark, did you finish the research paper for Professor Chen's class?"
          },
          {
            "speaker": "male",
            "segment_file": "1-09-b.ogg",
            "text": "Not yet. I'm having trouble finding reliable sources for my topic on renewable energy. The library database seems limited."
          },
          {
            "speaker": "female",
            "segment_file": "1-09-c.ogg",
            "text": "Have you tried accessing the digital archives? They have a lot of recent studies. Plus, you can request articles from other universities through interlibrary loan."
          },
          {
            "speaker": "male",
            "segment_file": "1-09-d.ogg",
            "text": "Really? How long does that usually take?"
          },
          {
            "speaker": "female",
            "segment_file": "1-09-e.ogg",
            "text": "About 3-5 days for most requests. Since your paper isn't due until next Friday, you should have plenty of time."
          }
        ],
        "pauses": [],
        "concat": {
          "segments": [
            "1-09-a.ogg",
            "1-09-b.ogg",
            "1-09-c.ogg",
            "1-09-d.ogg",
            "1-09-e.ogg"
          ],
          "output": "1-09.ogg"
        }
      },
      {
        "id": "1-10",
        "output": "1-10.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-10-a.ogg",
            "text": "I think my laptop is acting up again. It keeps freezing and shutting down randomly."
          },
          {
            "speaker": "female",
            "segment_file": "1-10-b.ogg",
            "text": "That's frustrating. Have you tried restarting it or running a virus scan?"
          },
          {
            "speaker": "male",
            "segment_file": "1-10-c.ogg",
            "text": "Yes, I did both, but it didn't help. I might need to take it to a repair shop."
          },
          {
            "speaker": "female",
            "segment_file": "1-10-d.ogg",
            "text": "That sounds like a good idea. They should be able to diagnose and fix the issue."
          }
        ],
        "pauses": [],
        "concat": {
          "segments": [
            "1-10-a.ogg",
            "1-10-b.ogg",
            "1-10-c.ogg",
            "1-10-d.ogg"
          ],
          "output": "1-10.ogg"
        }
      },
      {
        "id": "1-11",
        "output": "1-11.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-11.ogg",
            "text": "Attention, students. Due to the upcoming career fair next Wednesday, the main auditorium will be unavailable for regular classes from 8:00 AM to 6:00 PM. All classes normally held in the auditorium that day should be moved to alternative locations. Students should check with their professors for updated room assignments. The career fair will feature over 50 companies from various industries, including technology, healthcare, and finance. Registration for the career fair is still open until Monday evening."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "1-12",
        "output": "1-12.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "1-12.ogg",
            "text": "Today we're going to discuss biomimicry, the human practice of learning from and imitating things found in nature to create new innovations. This approach allows scientists and engineers to develop solutions that are both efficient and sustainable. One well-known example is the development of Velcro. The inventor was inspired by the way burrs from plants stuck to his dog's fur. By studying the tiny hooks on the burrs, he created a new fastening system. Another example is energy-efficient building design based on termite mounds. Termites create complex ventilation systems that maintain stable internal temperatures, which architects now emulate in human buildings. Biomimicry is also being applied to renewable energy. Researchers are developing wind turbine blades modeled after whale fins. The unique shape reduces drag and increases efficiency, leading to more effective wind energy capture. These innovations demonstrate how nature provides blueprints for solving modern technological challenges."
          }
        ],
        "pauses": [],
        "concat": null
      }
    ]
  },
  {
    "name": "pt1-speaking-m1",
    "blocks": [
      {
        "id": "3-01",
        "output": "3-01.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-01.ogg",
            "text": "Welcome to our bookstore café."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-02",
        "output": "3-02.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-02.ogg",
            "text": "New releases are displayed near the front windows."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-03",
        "output": "3-03.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-03.ogg",
            "text": "Coffee and pastries are available at the counter."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-04",
        "output": "3-04.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-04.ogg",
            "text": "Study areas are located on the second floor."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-05",
        "output": "3-05.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-05.ogg",
            "text": "Please keep your voice down in the quiet reading sections."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-06",
        "output": "3-06.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-06.ogg",
            "text": "Wi-Fi passwords are posted near each table."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-07",
        "output": "3-07.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-07.ogg",
            "text": "Book recommendations can be found on our monthly newsletter."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-08",
        "output": "3-08.ogg",
        "segments": [
          {
            "speaker": "female",
            "segment_file": "3-08.ogg",
            "text": "If customers need assistance, please direct them to the information desk."
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-09",
        "output": "3-09.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "3-09.ogg",
            "text": "Thank you for participating in our study today. I'm researching how students approach learning and studying. I'd like to ask you some questions. First, do you prefer studying alone or with other people? Why?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-10",
        "output": "3-10.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "3-10.ogg",
            "text": "Interesting. Many students struggle with managing their time effectively. Some find it helpful to create detailed schedules, while others prefer a more flexible approach. What methods do you use to organize your study time? How effective do you think your current system is?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-11",
        "output": "3-11.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "3-11.ogg",
            "text": "Good points. Let me ask about technology. Some educators believe that digital devices like laptops and tablets enhance learning, while others think they can be distracting. What's your opinion on using technology for studying? Can you give me specific examples from your own experience?"
          }
        ],
        "pauses": [],
        "concat": null
      },
      {
        "id": "3-12",
        "output": "3-12.ogg",
        "segments": [
          {
            "speaker": "male",
            "segment_file": "3-12.ogg",
            "text": "That's helpful. Finally, looking ahead, some people think that traditional classroom learning will eventually be replaced by online education. Do you believe online learning can be as effective as in-person classes? What are the advantages and disadvantages of each approach?"
          }
        ],
        "pauses": [],
        "concat": null
      }
    ]
  }
]''')

total = sum(len(g["blocks"]) for g in ALL_GROUPS)
print(f"Loaded {total} audio files across {len(ALL_GROUPS)} group(s)")
for g in ALL_GROUPS:
    print(f"  {g['name']}: {len(g['blocks'])} files")

In [ ]:
def generate_segment(text, voice):
    """Run Kokoro TTS and return audio as numpy array."""
    chunks = []
    for _, _, audio in pipeline(text, voice=voice):
        chunks.append(audio)
    if not chunks:
        print(f"  [warn] No audio generated for: {text[:50]}...")
        return np.zeros(SAMPLE_RATE, dtype=np.float32)
    return np.concatenate(chunks)

def make_silence(seconds):
    return np.zeros(int(SAMPLE_RATE * seconds), dtype=np.float32)

print("Ready")

In [ ]:
from IPython.display import display, Audio

group_outputs = {}  # name -> list of (output_name, audio_array)
counter = 0
total = sum(len(g["blocks"]) for g in ALL_GROUPS)

for group in ALL_GROUPS:
    name = group["name"]
    outputs = []
    for block in group["blocks"]:
        counter += 1
        file_id = block["id"]
        output_name = block["output"]
        segments = block["segments"]
        pauses = {p["after_segment"]: p["seconds"] for p in block["pauses"]}

        print(f"[{counter}/{total}] {name}/{file_id} -> {output_name}")

        seg_audios = []
        for si, seg in enumerate(segments):
            voice = VOICE_MAP.get(seg["speaker"], "af_heart")
            text = seg["text"]
            preview = text[:60] + ("..." if len(text) > 60 else "")
            print(f"  {si+1}/{len(segments)} {seg['speaker']} ({voice}): {preview}")
            audio = generate_segment(text, voice)
            seg_audios.append(audio)
            if si in pauses:
                seg_audios.append(make_silence(pauses[si]))

        if len(seg_audios) == 1:
            final = seg_audios[0]
        else:
            gap = make_silence(0.4)
            parts = []
            for i, a in enumerate(seg_audios):
                if i > 0: parts.append(gap)
                parts.append(a)
            final = np.concatenate(parts)

        outputs.append((output_name, final))
        print(f"  -> {len(final)/SAMPLE_RATE:.1f}s")

    group_outputs[name] = outputs

print(f"\nGenerated {counter} audio files")

In [ ]:
# Preview first audio from each group
for name, outputs in group_outputs.items():
    if outputs:
        fname, audio = outputs[0]
        print(f"{name}/{fname} ({len(audio)/SAMPLE_RATE:.1f}s)")
        display(Audio(data=audio, rate=SAMPLE_RATE, autoplay=False))

In [ ]:
# Convert to OGG and create zip files
zip_files = []

for name, outputs in group_outputs.items():
    out_dir = name
    wav_dir = os.path.join(out_dir, "_wav")
    os.makedirs(wav_dir, exist_ok=True)

    ogg_files = []
    for fname, audio in outputs:
        wav_path = os.path.join(wav_dir, fname.replace(".ogg", ".wav"))
        sf.write(wav_path, audio, SAMPLE_RATE)
        ogg_path = os.path.join(out_dir, fname)
        subprocess.run(
            ["ffmpeg", "-y", "-i", wav_path, "-c:a", "libvorbis", "-q:a", "6", ogg_path],
            capture_output=True
        )
        ogg_files.append(ogg_path)

    shutil.rmtree(wav_dir)

    zip_name = f"{name}.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for ogg in ogg_files:
            zf.write(ogg, os.path.basename(ogg))
    zip_files.append(zip_name)
    size = os.path.getsize(zip_name)
    print(f"{zip_name}: {len(ogg_files)} files, {size/1024:.0f} KB")

print(f"\nCreated {len(zip_files)} zip file(s)")

In [ ]:
# Download zip files
try:
    from google.colab import files
    for zf in zip_files:
        files.download(zf)
    print("Downloads started")
except ImportError:
    print("Not running on Colab. Files are in the current directory:")
    for zf in zip_files:
        print(f"  {zf}")